In [10]:
!pip install scikit-learn joblib

In [11]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [12]:
df = pd.read_csv('processed/flights_processed.csv')
print(df.shape)
print('Class distribution:')
print(df['delayed'].value_counts())
print('Delay rate:', df['delayed'].mean().round(3))

(560352, 8)
Class distribution:
delayed
0    430916
1    129436
Name: count, dtype: int64
Delay rate: 0.231


In [13]:
X = df[['airline', 'origin', 'destination', 'dep_hour', 'day_of_week', 'month', 'distance']]
y = df['delayed']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train size:', X_train.shape[0])
print('Test size:', X_test.shape[0])

Train size: 448281
Test size: 112071


In [14]:
categorical_features = ['airline', 'origin', 'destination']
numeric_features = ['dep_hour', 'day_of_week', 'month', 'distance']

preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('num', StandardScaler(), numeric_features)
])

In [15]:
# Baseline: Logistic Regression with class_weight balanced
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]

print('--- Logistic Regression (balanced) ---')
print('Accuracy:', accuracy_score(y_test, lr_preds))
print('ROC AUC:', roc_auc_score(y_test, lr_proba))
print(classification_report(y_test, lr_preds))

--- Logistic Regression (balanced) ---
Accuracy: 0.6006549419564383
ROC AUC: 0.6376228524635077
              precision    recall  f1-score   support

           0       0.83      0.60      0.70     86184
           1       0.31      0.60      0.41     25887

    accuracy                           0.60    112071
   macro avg       0.57      0.60      0.55    112071
weighted avg       0.71      0.60      0.63    112071



In [16]:
# Main model: Random Forest with class_weight balanced
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'))
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)
rf_proba = rf_pipeline.predict_proba(X_test)[:, 1]

print('--- Random Forest (balanced) ---')
print('Accuracy:', accuracy_score(y_test, rf_preds))
print('ROC AUC:', roc_auc_score(y_test, rf_proba))
print(classification_report(y_test, rf_preds))

--- Random Forest (balanced) ---
Accuracy: 0.6517743216353918
ROC AUC: 0.5844511177202466
              precision    recall  f1-score   support

           0       0.79      0.74      0.77     86184
           1       0.29      0.35      0.32     25887

    accuracy                           0.65    112071
   macro avg       0.54      0.55      0.54    112071
weighted avg       0.68      0.65      0.66    112071



In [17]:
# Feature importance from Random Forest
ohe_features = rf_pipeline.named_steps['preprocessor'].transformers_[0][1].get_feature_names_out(categorical_features).tolist()
all_features = ohe_features + numeric_features

importances = rf_pipeline.named_steps['classifier'].feature_importances_
feat_df = pd.DataFrame({'feature': all_features, 'importance': importances})
feat_df.sort_values('importance', ascending=False).head(15)

,feature,importance
712,dep_hour,0.299862
713,day_of_week,0.241813
715,distance,0.079040
3,airline_DL,0.005515
9,airline_WN,0.004907
0,airline_AA,0.004069
8,airline_UA,0.003976
248,origin_ORD,0.003451
543,destination_LAS,0.003254
612,destination_PHX,0.003225


In [18]:
# Save the best model (Random Forest)
joblib.dump(rf_pipeline, 'processed/model.pkl')
print('Model saved to processed/model.pkl')

Model saved to processed/model.pkl
